# 任意のモノドロミーにおける $\tau_1^\theta$ の独立検証

入力：`aAbBcCdDeEfF` からなる任意の語

In [ ]:
MONODROMY = 'DbCa'
SHOW_ROUTE_A_STEPS = True
SHOW_ROUTE_B_DETAILS = True

ALLOWED_LETTERS = set('aAbBcCdDeEfF')
invalid_letters = set(MONODROMY) - ALLOWED_LETTERS
if invalid_letters:
    raise ValueError(f'Unknown letters: {sorted(invalid_letters)}')

print('MONODROMY =', MONODROMY)

## Conventions

In [ ]:
from sage.all import (
    FreeGroup, ZZ, QQ, vector, matrix,
    identity_matrix, zero_matrix
)

%display latex

F = FreeGroup(4, 'x,y,z,w')
x, y, z, w = F.generators()
BASIS = (x, y, z, w)
GEN_NAMES = {1: 'x', 2: 'y', 3: 'z', 4: 'w'}

def comm(u, v):
    return u * v * (~u) * (~v)

def conj(u, v):
    return v * u * (~v)

bnd = comm(x, y) * comm(z, w)

X_H = vector(QQ, [1, 0, 0, 0])
Y_H = vector(QQ, [0, 1, 0, 0])
Z_H = vector(QQ, [0, 0, 1, 0])
W_H = vector(QQ, [0, 0, 0, 1])
H_STANDARD_BASIS = (X_H, Y_H, Z_H, W_H)

J = matrix(QQ, [
    [ 0,  1,  0, 0],
    [-1,  0,  0, 0],
    [ 0,  0,  0, 1],
    [ 0,  0, -1, 0],
])

WEDGE_PAIRS = (
    (0, 1), (0, 2), (0, 3),
    (1, 2), (1, 3), (2, 3)
)
TRIPLE_PAIRS = ((0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3))

def matrix_from_columns(columns, ring=QQ):
    return matrix(
        ring, len(columns[0]), len(columns),
        lambda i, j: columns[j][i]
    )

def wedge(u, v):
    u = vector(QQ, u)
    v = vector(QQ, v)
    return vector(QQ, [
        u[i]*v[j] - u[j]*v[i]
        for i, j in WEDGE_PAIRS
    ])

def wedge_action_matrix(A):
    A = A.change_ring(QQ)
    return matrix_from_columns([
        wedge(A.column(i), A.column(j))
        for i, j in WEDGE_PAIRS
    ], QQ)

## Route A — Dehn twist formula and cocycle composition

In [ ]:
def eta_coeff(eta, i, j):
    if i == j:
        return QQ(0)
    if i < j:
        return eta[WEDGE_PAIRS.index((i, j))]
    return -eta[WEDGE_PAIRS.index((j, i))]

def wedge_H_Lambda2(u, eta):
    u = vector(QQ, u)
    eta = vector(QQ, eta)
    return vector(QQ, [
        u[i] * eta_coeff(eta, j, k)
        - u[j] * eta_coeff(eta, i, k)
        + u[k] * eta_coeff(eta, i, j)
        for i, j, k in TRIPLE_PAIRS
    ])

def symplectic_pair(u, v):
    return vector(QQ, u).dot_product(J * vector(QQ, v))

def lambda3_to_hom(lam):
    columns = []
    for h in H_STANDARD_BASIS:
        eta = vector(QQ, 6)
        for coeff, (i, j, k) in zip(lam, TRIPLE_PAIRS):
            e_i = H_STANDARD_BASIS[i]
            e_j = H_STANDARD_BASIS[j]
            e_k = H_STANDARD_BASIS[k]
            eta += coeff * (
                symplectic_pair(h, e_i) * wedge(e_j, e_k)
                - symplectic_pair(h, e_j) * wedge(e_i, e_k)
                + symplectic_pair(h, e_k) * wedge(e_i, e_j)
            )
        columns.append(eta)
    return matrix_from_columns(columns, QQ)

def transvection_matrix(v, sign=1):
    v = vector(QQ, v)
    columns = [
        u + sign * symplectic_pair(v, u) * v
        for u in H_STANDARD_BASIS
    ]
    return matrix_from_columns(columns, QQ)

In [ ]:
# 理論側の入力：|c| と ell2(c)
CURVE_H = {
    'a': X_H,
    'b': -Y_H + W_H,
    'c': Z_H,
    'd': W_H,
    'e': W_H,
    'f': Y_H,
}

CURVE_ELL2 = {
    'a':  QQ(1)/2 * wedge(X_H, Y_H),
    'b': (QQ(1)/2 * wedge(X_H, Y_H)
          - QQ(1)/2 * wedge(Y_H, W_H)
          + QQ(1)/2 * wedge(Z_H, W_H)),
    'c':  QQ(1)/2 * wedge(Z_H, W_H),
    'd': -QQ(1)/2 * wedge(Z_H, W_H),
    'e':  wedge(X_H, Y_H) + QQ(1)/2 * wedge(Z_H, W_H),
    'f': -QQ(1)/2 * wedge(X_H, Y_H),
}

def positive_twist_theory(letter):
    v = CURVE_H[letter]
    ell_c = CURVE_ELL2[letter]
    A = transvection_matrix(v, sign=1)
    lambda3 = -wedge_H_Lambda2(v, ell_c)
    tau = lambda3_to_hom(lambda3)
    return {'A': A, 'lambda3': lambda3, 'tau': tau}

def twist_theory(letter):
    lower = letter.lower()
    positive = positive_twist_theory(lower)

    if letter.islower():
        return positive

    A = positive['A']
    A2 = wedge_action_matrix(A)
    A_inverse = A.inverse()
    tau_inverse = -A2.inverse() * positive['tau'] * A

    return {
        'A': A_inverse,
        'lambda3': None,
        'tau': tau_inverse,
    }

In [ ]:
def route_A(monodromy, show_steps=False):
    A_partial = identity_matrix(QQ, 4)
    tau_partial = zero_matrix(QQ, 6, 4)
    partial_word = ''

    # 右端から順に、letter o partial_word を作る。
    for letter in reversed(monodromy):
        generator = twist_theory(letter)
        A_letter = generator['A']
        tau_letter = generator['tau']

        # tau(letter o partial)
        tau_partial = (
            tau_letter
            + wedge_action_matrix(A_letter)
              * tau_partial
              * A_letter.inverse()
        )
        A_partial = A_letter * A_partial
        partial_word = letter + partial_word

        if show_steps:
            action_order = ' -> '.join(reversed(partial_word))
            print(f'partial word = {partial_word}')
            print(f'application order = {action_order}')
            print('A =')
            print(A_partial)
            print('tau =')
            print(tau_partial)
            print()

    return {'A': A_partial, 'tau': tau_partial}

result_A = route_A(MONODROMY, SHOW_ROUTE_A_STEPS)

print('Route A: final A =')
print(result_A['A'])
print('Route A: final tau =')
print(result_A['tau'])

## Route B — explicit word substitution on $\pi$

In [ ]:
# 自由群上の作用を生成元像として直接列挙
a_word = x
b_word = (~y) * conj(w, z)
c_word = z
d_word = w
e_word = bnd * w
f_word = y

ACTION = {
    'a': {'y': y * a_word},
    'A': {'y': y * (~a_word)},
    'b': {
        'x': x * b_word,
        'y': conj(y, ~b_word),
        'z': (~b_word) * z,
    },
    'B': {
        'x': x * (~b_word),
        'y': conj(y, b_word),
        'z': b_word * z,
    },
    'c': {'w': w * c_word},
    'C': {'w': w * (~c_word)},
    'd': {'z': z * (~d_word)},
    'D': {'z': z * d_word},
    'e': {
        'x': conj(x, ~e_word),
        'y': conj(y, ~e_word),
        'z': (~e_word) * z,
    },
    'E': {
        'x': conj(x, e_word),
        'y': conj(y, e_word),
        'z': e_word * z,
    },
    'f': {'x': x * (~f_word)},
    'F': {'x': x * f_word},
}

def substitute_word(element, action):
    result = F.one()
    for idx in element.Tietze():
        name = GEN_NAMES[abs(idx)]
        generator = F.gen(abs(idx) - 1)
        image = action.get(name, generator)
        result *= image if idx > 0 else ~image
    return result

def apply_monodromy(element, monodromy):
    result = element
    for letter in reversed(monodromy):
        result = substitute_word(result, ACTION[letter])
    return result

In [ ]:
def homology(element):
    result = vector(ZZ, 4)
    for idx in element.Tietze():
        i = abs(idx) - 1
        result[i] += 1 if idx > 0 else -1
    return result

ELL2_GENERATORS = {
    1: vector(QQ, [ QQ(1)/2, 0, 0, 0, 0, 0]),
    2: vector(QQ, [-QQ(1)/2, 0, 0, 0, 0, 0]),
    3: vector(QQ, [0, 0, 0, 0, 0,  QQ(1)/2]),
    4: vector(QQ, [0, 0, 0, 0, 0, -QQ(1)/2]),
}

def ell2_letter(idx):
    value = vector(QQ, ELL2_GENERATORS[abs(idx)])
    return value if idx > 0 else -value

def ell2_word(element):
    prefix_H = vector(QQ, 4)
    result = vector(QQ, 6)

    for idx in element.Tietze():
        letter_H = vector(QQ, 4)
        i = abs(idx) - 1
        letter_H[i] = 1 if idx > 0 else -1

        result += (
            ell2_letter(idx)
            + QQ(1)/2 * wedge(prefix_H, letter_H)
        )
        prefix_H += letter_H

    return result

assert ell2_word(bnd) == vector(QQ, [1, 0, 0, 0, 0, 1])

In [ ]:
def route_B(monodromy, show_details=False):
    images = [
        apply_monodromy(g, monodromy)
        for g in BASIS
    ]

    H_images = [homology(image) for image in images]
    A = matrix_from_columns(H_images, ZZ)
    A2 = wedge_action_matrix(A)

    ell_images = [ell2_word(image) for image in images]
    A2_ell_generators = [A2 * ell2_word(g) for g in BASIS]
    deltas = [
        ell_image - transformed_ell
        for ell_image, transformed_ell
        in zip(ell_images, A2_ell_generators)
    ]

    D = matrix_from_columns(deltas, QQ)
    tau = D * A.change_ring(QQ).inverse()

    assert A.transpose() * J * A == J
    assert apply_monodromy(bnd, monodromy) == bnd

    if show_details:
        for name, image, H_image, ell_image, delta in zip(
            ('x', 'y', 'z', 'w'),
            images, H_images, ell_images, deltas
        ):
            print(f'phi({name}) = {image}')
            print(f'|phi({name})| = {H_image}')
            print(f'ell2(phi({name})) = {ell_image}')
            print(f'Delta_{name} = {delta}')
            print()

        print('A =')
        print(A)
        print('Lambda^2 A =')
        print(A2)
        print('D =')
        print(D)
        print('tau = D A^(-1) =')
        print(tau)

    return {
        'images': images,
        'A': A.change_ring(QQ),
        'A2': A2,
        'D': D,
        'tau': tau,
    }

result_B = route_B(MONODROMY, SHOW_ROUTE_B_DETAILS)

## Comparison

In [ ]:
A_equal = result_A['A'] == result_B['A']
tau_equal = result_A['tau'] == result_B['tau']

print('A: Route A == Route B :', A_equal)
print('tau: Route A == Route B :', tau_equal)

if not A_equal:
    print('A difference =')
    print(result_A['A'] - result_B['A'])

if not tau_equal:
    print('tau difference =')
    print(result_A['tau'] - result_B['tau'])

assert A_equal
assert tau_equal